# save datasets

Notebook de sauvegarde des tables PostgreSQL `grocery_db` au format CSV.


In [4]:
import pandas as pd
import psycopg2
from pathlib import Path

DB_CONFIG = {
    "dbname": "grocery_db",
    "user": "postgres",
    "password": "2002",
    "host": "localhost",
    "port": 5432,
}

TABLES = [
    "dim_category",
    "dim_customer",
    "dim_employee",
    "dim_product",
    "fact_sales",
]

OUTPUT_DIR = Path.cwd() / "csv_exports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def get_connection():
    return psycopg2.connect(**DB_CONFIG)


def save_table_csv(table_name: str):
    with get_connection() as conn:
        df = pd.read_sql(f"SELECT * FROM {table_name}", conn)

    if table_name == "fact_sales" and "totalprice" in df.columns:
        df = df.drop(columns=["totalprice"])
        print("Column 'totalprice' removed from fact_sales export.")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])

    output_path = OUTPUT_DIR / f"{table_name}.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved {table_name} → {output_path}")
    return df


## Suppression définitive de la colonne `totalprice`
Cette cellule supprime la colonne `totalprice` de la table PostgreSQL `fact_sales` si elle existe, puis recharge et exporte `fact_sales` sans cette colonne.

In [5]:
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(
            "ALTER TABLE fact_sales DROP COLUMN IF EXISTS totalprice;"
        )
    conn.commit()

print("La colonne 'totalprice' a été supprimée de la table fact_sales si elle existait.")

# Réexporter fact_sales sans totalprice
saved_fact_sales = save_table_csv("fact_sales")
print(f"fact_sales exported with {len(saved_fact_sales.columns)} columns.")

La colonne 'totalprice' a été supprimée de la table fact_sales si elle existait.


La colonne 'totalprice' a été supprimée de la table fact_sales si elle existait.


C:\Users\lenovo\AppData\Local\Temp\ipykernel_30624\1671329070.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {table_name}", conn)


La colonne 'totalprice' a été supprimée de la table fact_sales si elle existait.


C:\Users\lenovo\AppData\Local\Temp\ipykernel_30624\1671329070.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {table_name}", conn)


KeyboardInterrupt: 

In [ ]:
# Exporter toutes les tables PostgreSQL en CSV
saved_tables = {}
for table in TABLES:
    try:
        saved_tables[table] = save_table_csv(table)
    except Exception as exc:
        print(f"Erreur pour {table}: {exc}")

print("\nExport terminé. Fichiers générés dans :", OUTPUT_DIR)

C:\Users\lenovo\AppData\Local\Temp\ipykernel_30624\3409468772.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {table_name}", conn)


Saved dim_category → c:\Users\lenovo\Desktop\BI Project\bi\datasets\csv_exports\dim_category.csv
Saved dim_customer → c:\Users\lenovo\Desktop\BI Project\bi\datasets\csv_exports\dim_customer.csv
Saved dim_employee → c:\Users\lenovo\Desktop\BI Project\bi\datasets\csv_exports\dim_employee.csv
Saved dim_product → c:\Users\lenovo\Desktop\BI Project\bi\datasets\csv_exports\dim_product.csv
Saved fact_sales → c:\Users\lenovo\Desktop\BI Project\bi\datasets\csv_exports\fact_sales.csv

Export terminé. Fichiers générés dans : c:\Users\lenovo\Desktop\BI Project\bi\datasets\csv_exports


In [ ]:
# Aperçu rapide des fichiers exportés
for table, df in saved_tables.items():
    print(f"\n=== {table} ({len(df)} lignes) ===")
    display(df.head())


=== dim_category (11 lignes) ===


,categoryid,categoryname
0,1,Confections
1,2,Shell fish
2,3,Cereals
3,4,Dairy
4,5,Beverages



=== dim_customer (98759 lignes) ===


,customerid,customerfirstname,middleinitial,customerlastname,address,cityid,city,zipcode,countryid,country,countrycode
0,1,Stefanie,Y,Frye,97 Oak Avenue,79,Oklahoma,40472,32,United States,AR
1,2,Sandy,T,Kirby,52 White First Freeway,96,Pittsburgh,14257,32,United States,AR
2,3,Lee,T,Zhang,921 White Fabien Avenue,55,Houston,95800,32,United States,AR
3,4,Regina,S,Avery,75 Old Avenue,40,Cleveland,51352,32,United States,AR
4,5,Daniel,S,Mccann,283 South Green Hague Avenue,2,Buffalo,17420,32,United States,AR



=== dim_employee (23 lignes) ===


,employeeid,employeefirstname,employeelastname,birthdate,gender,hiredate,city,cityid
0,1,Nicole,Fuller,1981-03-07,F,2011-06-20,Tampa,74
1,2,Christine,Palmer,1968-01-25,F,2011-04-27,Chicago,3
2,3,Pablo,Cline,1963-02-09,M,2012-03-30,Bakersfield,41
3,4,Darnell,Nielsen,1989-02-06,M,2014-03-06,Mobile,62
4,5,Desiree,Stuart,1963-05-03,F,2014-11-16,Anchorage,68



=== dim_product (452 lignes) ===


,productid,productname,price,categoryid,class,modifydate,resistant,isallergic,vitalitydays,categoryname
0,1,Flour - Whole Wheat,74.3,3,Medium,2018-02-16,Durable,Unknown,0.0,Cereals
1,2,Cookie Chocolate Chip With,91.2,3,Medium,2017-02-12,Unknown,Unknown,0.0,Cereals
2,3,Onions - Cippolini,9.1,9,Medium,2018-03-15,Weak,False,111.0,Poultry
3,4,"Sauce - Gravy, Au Jus, Mix",54.3,9,Medium,2017-07-16,Durable,Unknown,0.0,Poultry
4,5,Artichokes - Jerusalem,65.5,2,Low,2017-08-16,Durable,True,27.0,Shell fish



=== fact_sales (6690599 lignes) ===


,salesid,employeeid,customerid,productid,date,quantity,discount,totalprice,transactionnumber,time
0,3846545,13,31108,214,2022-09-18,8,0.0,0.0,O5OBGH6TJPIOAXYHREZQ,23:00:07
1,3846546,12,19124,330,2021-06-28,5,0.0,0.0,JVO34H61TBGEVG3ZR8MM,21:40:23
2,3846547,14,20421,67,2018-03-17,6,0.0,0.0,G3CFDEFCHNESG5907B0Z,13:41:48
3,3846548,6,14125,1,2019-01-26,4,0.0,0.0,QD19FGVHZ5D4H2SBOQ3Q,04:55:05
4,3846549,10,86300,18,2018-07-03,22,0.0,0.0,CN2MFBTZJWLMRQPB5YXK,09:29:09


## Résultat

Le notebook exporte les tables suivantes dans `datasets/csv_exports/` :
- `dim_category.csv`
- `dim_customer.csv`
- `dim_employee.csv`
- `dim_product.csv`
- `fact_sales.csv`

Exécute simplement la cellule d’export pour générer tous les CSV.